# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url) 

df.columns = df.columns.str.lower().str.replace(" ", "_")

df = df.drop(columns=["unnamed:_0"])

print(df.shape)

(10910, 25)


In [ ]:
# condición 1: reclamaciones por debajo de 1000
reclamacion_baja = df["total_claim_amount"] < 1000

# condición 2: respondió "Yes" a la última campaña
respondio_si = df["response"] == "Yes"

# me quedo con las filas que cumplen las dos
df_yes_bajo = df[reclamacion_baja & respondio_si]

print(df_yes_bajo.shape)
df_yes_bajo[["customer", "response", "total_claim_amount"]].head()

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [ ]:
# me quedo solo con los clientes que respondieron "Yes"
df_yes = df[df["response"] == "Yes"]

# agrupo por tipo de póliza y género, y calculo la media de prima, CLV y reclamaciones
medias_yes = df_yes.groupby(["policy_type", "gender"])[
    ["monthly_premium_auto", "customer_lifetime_value", "total_claim_amount"]
].mean().round(2)

medias_yes

monthly_premium_auto  customer_lifetime_value  \
policy_type    gender                                                  
Corporate Auto F                      94.30                  7712.63   
               M                      92.19                  7944.47   
Personal Auto  F                      99.00                  8339.79   
               M                      91.09                  7448.38   
Special Auto   F                      92.31                  7691.58   
               M                      86.34                  8247.09   

                       total_claim_amount  
policy_type    gender                      
Corporate Auto F                   433.74  
               M                   408.58  
Personal Auto  F                   452.97  
               M                   457.01  
Special Auto   F                   453.28  
               M                   429.53

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [ ]:
# agrupo por estado y cuento cuántos clientes distintos hay en cada uno
clientes_estado = df.groupby("state")["customer"].nunique()

# ordeno de mayor a menor
clientes_estado = clientes_estado.sort_values(ascending=False)

# filtro: me quedo solo con los estados que tienen más de 500 clientes
clientes_estado_500 = clientes_estado[clientes_estado > 500]

clientes_estado_500

state
California    3150
Oregon        2601
Arizona       1703
Nevada         882
Washington     798
Name: customer, dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [ ]:
# agrupo por nivel de estudios y género, miro el CLV y calculo el máximo, mínimo y mediana
clv_stats = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(
    ["max", "min", "median"]
).round(2)

clv_stats

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [ ]:
# convierto el texto de la fecha a fecha real; el formato es mes/día/año de 2 cifras
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y")

# de cada fecha me quedo solo con el número de mes (1 = enero, 2 = febrero...)
df["month"] = df["effective_to_date"].dt.month

df["month"].value_counts()

month
1    5818
2    5092
Name: count, dtype: int64

In [ ]:
# filas = estados, columnas = meses, y en cada celda cuento cuántas pólizas hay
polizas_estado_mes = df.pivot_table(
    index="state",
    columns="month",
    values="policy",
    aggfunc="count"   # count cuenta filas no vacías: cada fila es una póliza vendida
)

polizas_estado_mes

month,1,2
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# 1. cuento las pólizas totales por estado
polizas_por_estado = df.groupby("state")["policy"].count()

# 2. me quedo con los nombres de los 3 estados con más pólizas
top3_estados = polizas_por_estado.sort_values(ascending=False).head(3).index

# 3. de la tabla del ejercicio 5, me quedo solo con esas 3 filas
polizas_top3 = polizas_estado_mes.loc[top3_estados]

polizas_top3

month,1,2
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929
